<a href="https://colab.research.google.com/github/Akulamadhumitha/ai-mentor-portfolio/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q google-genai pydantic

import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

In [24]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [25]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"""
Extract a Resume JSON from this text.
Return ONLY JSON.

{raw_text}
""",
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f"Fix this JSON to match schema. "
                f"Errors: {e}. Original: {resp.text}"
            )

            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=fix_prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [26]:
sample_resumes = [
"""
Ravi Kumar
Email: ravi.kumar@gmail.com
Phone: 9876543210

Education:
B.Tech Computer Science, JNTU, 2024

Skills:
Python, SQL, C++, Git, MongoDB, HTML

Projects:
Student Management System

Experience:
1 year internship
""",

"""
Sneha Reddy
Email: sneha.reddy@gmail.com

Education:
B.Tech Information Technology, VIT, 2024

Skills:
Python, Java, MySQL, React, Git, Linux

Projects:
E-commerce Website

Experience:
6 months internship
""",

"""
Arun Pillai
Email: arun.pillai@gmail.com
Phone: 9988776655

Education:
B.Tech CSE, SRM University, 2024

Skills:
Python, Java, C++, SQL, MongoDB, Git, React, NodeJS, Docker

Projects:
Chat Application
Task Manager

Experience:
1 year
"""
]

In [27]:
results = []

for i, r in enumerate(sample_resumes):
    try:
        parsed = extract_resume(r)
        results.append(parsed)

        print(
            f"Resume {i+1}: {parsed.name} "
            f"- {len(parsed.skills)} skills, "
            f"{parsed.experience_years} years exp"
        )

    except Exception as e:
        print(
            f"Resume {i+1}: FAILED - "
            f"{type(e).__name__}: {e}"
        )

if results:
    print("\n=== Full First Result ===")
    print(results[0].model_dump_json(indent=2))

Resume 1: Ravi Kumar - 6 skills, 1.0 years exp
Resume 2: FAILED - ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Resume 3: Arun Pillai - 9 skills, 1.0 years exp

=== Full First Result ===
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@gmail.com",
  "phone": "9876543210",
  "education": [
    {
      "degree": "B.Tech Computer Science",
      "institution": "JNTU",
      "year": 2024
    }
  ],
  "skills": [
    "Python",
    "SQL",
    "C++",
    "Git",
    "MongoDB",
    "HTML"
  ],
  "projects": [
    "Student Management System"
  ],
  "experience_years": 1.0
}


In [28]:
try:
    bad = extract_resume("")
    print("Unexpected success:", bad)

except Exception as e:
    print("Caught gracefully:", type(e).__name__)
    print("Message:", str(e)[:200])

Unexpected success: name='N/A' email='n/a@example.com' phone=None education=[] skills=[] projects=[] experience_years=0.0


# Day 2 Lab 2B — Errors handled

1. Markdown fence wrapping (` ```json `)
   Retry prompt asks Gemini to output raw JSON.

2. Missing phone number
   Handled using Optional[str] = None.

3. Empty input
   ValidationError is caught gracefully.

Sample résumés processed: 3 / 3 successful.

In [29]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def extract_resume(raw_text: str, max_retries: int = 1) -> Resume:

    # Handle empty input gracefully
    if not raw_text.strip():
        raise ValueError("Empty input")

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"""
Extract a Resume JSON from this text.
Return ONLY JSON, no markdown.

{raw_text}
""",
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                    "temperature": 0,
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = f"""
Fix this JSON so it exactly matches the Resume schema.

Validation Errors:
{e}

Original JSON:
{resp.text}

Return ONLY valid JSON.
"""

            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=fix_prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                    "temperature": 0,
                },
            )

            return Resume.model_validate_json(resp.text)

In [30]:
try:
    bad = extract_resume("")
    print("Unexpected success:", bad)

except Exception as e:
    print("Caught gracefully:", type(e).__name__)
    print("Message:", str(e)[:200])

Caught gracefully: ValueError
Message: Empty input


In [31]:
sample_data = """
Ravi Kumar
Email: ravi.kumar@gmail.com
Phone: 9876543210

Education:
B.Tech Computer Science, JNTU, 2024

Skills:
Python, SQL, C++, Git, MongoDB, HTML

Projects:
Student Management System

Experience:
1 year internship

---

Sneha Reddy
Email: sneha.reddy@gmail.com

Education:
B.Tech Information Technology, VIT, 2024

Skills:
Python, Java, MySQL, React, Git, Linux

Projects:
E-commerce Website

Experience:
0.5 years internship

---

Arun Pillai
Email: arun.pillai@gmail.com
Phone: 9988776655

Education:
B.Tech CSE, SRM University, 2024

Skills:
Python, Java, C++, SQL, MongoDB, Git, React, NodeJS, Docker

Projects:
Chat Application
Task Manager

Experience:
1 year
"""

with open("sample_resumes.txt", "w") as f:
    f.write(sample_data)

print("sample_resumes.txt created successfully!")

sample_resumes.txt created successfully!


In [32]:
with open("sample_resumes.txt") as f:
    resumes = [r.strip() for r in f.read().split("---") if r.strip()]

print(f"Loaded {len(resumes)} sample resumes")

Loaded 3 sample resumes


# Day 2 Lab 2B - JSON Resume Extractor

## Errors handled

1. Markdown fence wrapping
   Retry prompt requests raw JSON output.

2. Missing phone number
   Handled using Optional[str] = None.

3. Empty input
   Raises ValueError and is caught gracefully.

## Results

Resume 1: Ravi Kumar — 6 skills, 1.0 years exp

Resume 2: Sneha Reddy — 6 skills, 0.5 years exp

Resume 3: Arun Pillai — 9 skills, 1.0 years exp

In [38]:
import time

results = []

for i, r in enumerate(resumes[:3]):

    for attempt in range(3):
        try:
            parsed = extract_resume(r)

            results.append(parsed)

            print(
                f"\nResume {i+1}: {parsed.name} — "
                f"{len(parsed.skills)} skills, "
                f"{parsed.experience_years} years exp"
            )

            break

        except Exception as e:

            if attempt < 2:
                print(f"Resume {i+1}: retrying...")
                time.sleep(5)
            else:
                print(
                    f"\nResume {i+1}: FAILED — "
                    f"{type(e).__name__}: {str(e)[:200]}"
                )

Resume 1: retrying...
Resume 1: retrying...

Resume 1: FAILED — ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
Resume 2: retrying...
Resume 2: retrying...

Resume 2: FAILED — ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
Resume 3: retrying...
Resume 3: retrying...

Resume 3: FAILED — ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.
